In [ ]:
import json
import os
import urllib.error
import urllib.parse
import urllib.request
from datetime import date
from decimal import Decimal
from getpass import getpass
from pathlib import Path

import pandas as pd

In [ ]:
REFRESH = False
BASE_URL = 'https://api.census.gov/data/timeseries/intltrade/imports/enduse'
START_MONTH = '2016-01'
END_MONTH = '2025-07'
COUNTRIES = {
    '2010': 'MEXICO',
    '5490': 'THAILAND',
    '5520': 'VIETNAM',
    '5570': 'MALAYSIA',
    '5590': 'SINGAPORE',
    '5700': 'CHINA',
    '5820': 'HONG KONG',
    '5830': 'TAIWAN'
}
COLUMNS = [
    'date', 'country', 'GEN_VAL_MO', 'YEAR', 'MONTH', 'cty_code',
    'I_ENDUSE', 'COMM_LVL', 'source_last_update'
]
ROOT = Path.cwd()
INPUT_FILES = ['census_imports_api_snapshot.json', 'census_imports_monthly_totals.csv']
DATA_DIR = next(
    (directory for directory in [ROOT / 'data', ROOT.parent / 'data', ROOT]
     if any((directory / name).is_file() for name in INPUT_FILES)),
    ROOT.parent / 'data' if ROOT.name == 'scripts' else ROOT / 'data'
)
PROJECT_DIR = DATA_DIR.parent if DATA_DIR.name == 'data' else ROOT
RESULTS_DIR = PROJECT_DIR / 'results'
SNAPSHOT_PATH = DATA_DIR / INPUT_FILES[0]
CSV_PATH = DATA_DIR / INPUT_FILES[1]

In [ ]:
def fetch_census(api_key):
    responses = []
    parameters = []
    for code in COUNTRIES:
        params = {
            'get': 'CTY_NAME,GEN_VAL_MO,I_ENDUSE,COMM_LVL,LAST_UPDATE',
            'CTY_CODE': code,
            'I_ENDUSE': '-',
            'time': f'from {START_MONTH} to {END_MONTH}'
        }
        parameters.append(params.copy())
        query = urllib.parse.urlencode(dict(params, key=api_key))
        request = urllib.request.Request(
            BASE_URL + '?' + query,
            headers={'User-Agent': 'Census-replication/1.0'}
        )
        try:
            with urllib.request.urlopen(request, timeout=60) as response:
                payload = json.loads(response.read())
        except urllib.error.HTTPError as error:
            raise RuntimeError(f'Census HTTP {error.code}; CTY_CODE={code}') from None
        except Exception:
            raise RuntimeError(f'Census response error; CTY_CODE={code}') from None
        if not isinstance(payload, list) or len(payload) < 2 or not isinstance(payload[0], list):
            raise ValueError(f'Census response format; CTY_CODE={code}')
        header = payload[0]
        if any(not isinstance(row, list) or len(row) != len(header) for row in payload[1:]):
            raise ValueError(f'Census response columns; CTY_CODE={code}')
        responses.append({
            'country_code': code,
            'header': header,
            'rows': [dict(zip(header, row)) for row in payload[1:]]
        })
    return {
        'endpoint': BASE_URL,
        'retrieved_date': date.today().isoformat(),
        'query_parameters_without_credentials': parameters,
        'responses': responses
    }

def snapshot_frame(snapshot):
    rows = []
    for response in snapshot['responses']:
        for record in response['rows']:
            if record['CTY_CODE'] != response['country_code']:
                raise ValueError('CTY_CODE')
            month = record['time']
            rows.append({
                'date': month + '-01',
                'country': record['CTY_NAME'],
                'GEN_VAL_MO': record['GEN_VAL_MO'],
                'YEAR': month[:4],
                'MONTH': month[5:7],
                'cty_code': record['CTY_CODE'],
                'I_ENDUSE': record['I_ENDUSE'],
                'COMM_LVL': record['COMM_LVL'],
                'source_last_update': record['LAST_UPDATE']
            })
    return pd.DataFrame(rows, columns=COLUMNS)

def validate_totals(frame):
    missing = set(COLUMNS).difference(frame.columns)
    if missing:
        raise ValueError(f'Columns: {sorted(missing)}')
    frame = frame[COLUMNS].copy()
    for column in ['date', 'country', 'cty_code', 'I_ENDUSE', 'COMM_LVL', 'source_last_update']:
        frame[column] = frame[column].astype(str)
    for column in ['GEN_VAL_MO', 'YEAR', 'MONTH']:
        frame[column] = frame[column].map(lambda value: int(str(value)))
    months = [str(period) + '-01' for period in pd.period_range(START_MONTH, END_MONTH, freq='M')]
    expected = {(code, month) for code in COUNTRIES for month in months}
    keys = list(zip(frame['cty_code'], frame['date']))
    if len(keys) != len(set(keys)) or set(keys) != expected:
        raise ValueError('Country-month coverage or duplicates')
    if not frame['I_ENDUSE'].eq('-').all() or not frame['COMM_LVL'].eq('-').all():
        raise ValueError('I_ENDUSE or COMM_LVL')
    if not frame['country'].eq(frame['cty_code'].map(COUNTRIES)).all():
        raise ValueError('Country names')
    if frame['GEN_VAL_MO'].lt(0).any():
        raise ValueError('Negative GEN_VAL_MO')
    dates = pd.to_datetime(frame['date'], format='%Y-%m-%d')
    if not frame['YEAR'].eq(dates.dt.year).all() or not frame['MONTH'].eq(dates.dt.month).all():
        raise ValueError('YEAR or MONTH')
    return frame.sort_values(['cty_code', 'date']).reset_index(drop=True)

In [ ]:
snapshot = None
if REFRESH:
    api_key = os.environ.get('CENSUS_API_KEY', '').strip() or getpass('CENSUS_API_KEY: ').strip()
    if not api_key:
        raise ValueError('CENSUS_API_KEY')
    snapshot = fetch_census(api_key)
    del api_key
    df = snapshot_frame(snapshot)
elif SNAPSHOT_PATH.is_file():
    snapshot = json.loads(SNAPSHOT_PATH.read_text(encoding='utf-8'))
    df = snapshot_frame(snapshot)
elif CSV_PATH.is_file():
    df = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False)
else:
    raise FileNotFoundError(', '.join(INPUT_FILES))

df = validate_totals(df)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
if REFRESH:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    SNAPSHOT_PATH.write_text(json.dumps(snapshot, indent=2), encoding='utf-8')
    df.to_csv(CSV_PATH, index=False, lineterminator='\r\n')
df.to_csv(RESULTS_DIR / 'census_imports_monthly_totals.csv', index=False, lineterminator='\r\n')

In [ ]:
statistics = []
for code, group in df.groupby('cty_code', sort=True):
    total = sum(int(value) for value in group['GEN_VAL_MO'])
    mean = Decimal(total) / Decimal(len(group))
    billions = mean / Decimal(10 ** 9)
    statistics.append({
        'cty_code': code,
        'country': COUNTRIES[code],
        'months': len(group),
        'first_month': group['date'].min()[:7],
        'last_month': group['date'].max()[:7],
        'total_imports_usd': total,
        'mean_monthly_imports_usd': f'{mean:.6f}',
        'mean_monthly_imports_billion_usd': f'{billions:.9f}',
        'mean_monthly_imports_billion_usd_1dp': f'{billions:.1f}'
    })
statistics = pd.DataFrame(statistics)
statistics.to_csv(RESULTS_DIR / 'census_monthly_means.csv', index=False)
print(statistics.to_string(index=False))